In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".")  
DATA_MODEL = BASE_DIR / "data_model"

returns = pd.read_csv(DATA_MODEL / "fact_asset_returns.csv")
returns["price_date"] = pd.to_datetime(returns["price_date"])

# -------------------------
# 1. Static correlation matrix (full period)
# -------------------------

# Pivot to wide format: rows = date, cols = asset_symbol, values = daily_return
wide = returns.pivot_table(
    index="price_date",
    columns="asset_symbol",
    values="daily_return"
)

# Drop rows where all returns are NaN
wide = wide.dropna(how="all")

# Compute correlation matrix
corr_matrix = wide.corr()

# Convert to long format for Power BI
corr_long = corr_matrix.reset_index().melt(
    id_vars="asset_symbol",
    var_name="asset_symbol_2",
    value_name="correlation_value"
)
corr_long = corr_long.rename(columns={"asset_symbol": "asset_symbol_1"})

# -------------------------
# 2. Rolling 90-day correlation (BTC vs SPY, ETH vs SPY)
# -------------------------

pairs = [("BTC-USD", "SPY"), ("ETH-USD", "SPY")]
rolling_list = []

window = 90  # 90-day rolling

for a1, a2 in pairs:
    sub = wide[[a1, a2]].dropna(how="any")

    # rolling corr: series of corr between the two columns
    roll_corr = (
        sub[a1]
        .rolling(window=window)
        .corr(sub[a2])
        .dropna()
    )

    temp = roll_corr.reset_index()
    temp.columns = ["window_end_date", "correlation_value"]
    temp["pair_name"] = f"{a1} vs {a2}"
    temp["window_length_days"] = window

    rolling_list.append(temp)

rolling_corr = pd.concat(rolling_list, ignore_index=True)

# -------------------------
# Save outputs for Power BI
# -------------------------
corr_long.to_csv(DATA_MODEL / "fact_correlations.csv", index=False)
rolling_corr.to_csv(DATA_MODEL / "fact_rolling_correlations.csv", index=False)

print("Saved:")
print(" -", DATA_MODEL / "fact_correlations.csv")
print(" -", DATA_MODEL / "fact_rolling_correlations.csv")


Saved:
 - data_model\fact_correlations.csv
 - data_model\fact_rolling_correlations.csv
